# Data Cleanup Pipeline

Comprehensive record of all modifications applied to the initial dataset (`Everything merged 18.12._new.xlsx`).

All logic lives in the `pca_mri` library. 

## Acronyms & nomenclature
* `Rec`: Réception (ou enregistrement).
* `Inv`: Investigation (ou examen).

## Sample size
N = 110 patients, 71 columns in the raw file.

## Column naming convention (post-cleanup)
| Prefix | Domain |
|--------|--------|
| `tx-` | Treatment variables |
| `psa-` | PSA / CAPRA variables |
| `mri_N-` | Serial MRI visits (N = 1–4) |
| `biopsy-` | Recurrence-investigation biopsy |
| `pet-` | PET scan |
| `bf-` | Biochemical failure |

In [ ]:
import sys
sys.path.insert(0, '..')  # make pca_mri importable when running from repo root
import pandas as pd
import numpy as np

from pca_mri import io
from pca_mri.preprocessing import columns, patients, features

## 1. Load raw dataset

In [ ]:
df = pd.read_excel('Everything merged 18.12._new.xlsx')
print(df.shape)
df.head()

## 2. Remove empty columns

Drops all columns that are entirely NaN.

Known columns removed in the original dataset:  
`['Column8', 'Column12', 'Column15', 'Column18', 'Column19', '4th mri date', 'Column25', 'RecInvbiopsie.PSA', 'all patients.causedeces']`

In [ ]:
print(f'Shape before: {df.shape}')
df, dropped_empty = columns.drop_empty_columns(df)
print(f'Dropped {len(dropped_empty)} empty columns: {dropped_empty}')
print(f'Shape after: {df.shape}')

## 3. Remove duplicate columns

In [ ]:
# same_cols,remove_cols = columns.drop_duplicate_columns(df)
# print(remove_cols)
# print(same_cols)
print(f"before: {df.shape}")
remove_cols_final = ['RecInvbiopsie.PatientID', 'all patients.PatientId',
    'pet.PatientID','date treatment.PatientId', 'date treatment.TypeTX']
df = df.drop(columns=remove_cols_final)
print(f"after: {df.shape}")

## 4. Rename columns

Applies the project-standard naming convention defined in  
`pca_mri.preprocessing.columns.DEFAULT_COL_MAP`.

In [ ]:
print(f"before: {df.shape}")
df = columns.rename_columns(df)
print(f"after: {df.shape}")

## 5. Reorder columns

Groups columns logically: `patient_id` → `tx-*` → `psa-*` → `mri_N-*` → `biopsy-*` → `pet-*` → outcome dates.

In [ ]:
print(f"before: {df.shape}")
df = columns.reorder_columns(df)
print(f"after: {df.shape}")

## 6. Flag duplicate & special patients

### A. Duplicate patients
Known duplicates: `2060194`, `2054846`, `2205490`.
* `2205490` had 2 negative biopsies (2015-08-17 00:00:00, 2016-08-25 00:00:00). Deleted row 90 (kept 91)
* `2054846` had 2 negative biopsies (2014-09-30 00:00:00, 2016-02-15 00:00:00). Deleted row 105 (kept 106).
* `2060194` had 2 negative biopsies (2014-10-21 00:00:00, 2015-07-03 00:00:00). Deleted row 107 (kept 108).

In [ ]:
print(f"before: {df.shape}")
df = patients.flag_duplicate_patients(df) # adds `is_duplicate` boolean column
print(f"after: {df.shape}")
print('Duplicate patients:')
display(df[df['is_duplicate']][['patient_id', 'mri_1-result', 'biopsy-result', 'biopsy-date']])

In [ ]:
# df.loc[[90, 91]].T, df.loc[[105, 106]].T, df.loc[[105, 106]].T
df.loc[91, "biopsy-date"] = pd.Timestamp("2018-10-15")
df.loc[91, "biopsy-result"] = "Positif"

df = df.drop([90, 105, 107])
print(f"after: {df.shape}")

### B. Converter patients
Patients whose MRI changed from positive → negative over follow-up:
`5519879`, `7049838`, `7051280`, `7124048`.
Strategy: include first, then run sensitivity analysis excluding them (see analysis notebook).

In [ ]:
df = patients.flag_converter_patients(df)
print('Converter patients (positive → negative MRI):')
display(df[df['is_converter']][['patient_id', 'mri_1-result', 'mri_2-result', 'mri_3-result', 'mri_4-result', 'biopsy-result']])

### C. Missing biopsy results
2 subjects (`304665`, `5028002`) had missing biopsy "result" data but there was an associated biopsy date and associated biopsy Gleason score...

In [ ]:
print(df['biopsy-result'].isna().sum())
print(df['biopsy-date'].isna().sum())

In [ ]:
mask = df["biopsy-result"].isna() & df["biopsy-date"].notna() 
df.loc[mask, "biopsy-result"] = "Positif"

In [ ]:
print(df['biopsy-result'].isna().sum())
print(df['biopsy-date'].isna().sum())

## 7. Add derived features

* `tx-biopsy_positive_ratio`: positive samples / samples taken at time of Dx (≈ time Tx).
* `bf-time_to_bf-days`: days from treatment to biochemical failure (ASSUMPTION: patients with no "bf date" had no biochemical failure).
*  `rec_mri-{index, date, time_to_rec-days}`: Finds the first positive MRI for each patient and adds:
    + index of 1st positive MRI (0 if no positive MRI);
    + date of the first positive MRI (NaT if no positive MRI);
    + days from treatment to first positive MRI (0 if no positive MRI).
* `biopsy-time_to_biopsy-days`: days from treatment to recurrence-investigation biopsy.
* `bf_to_rec_mri-days`: days from biochemical failure to first positive MRI. Convention:
    + Positive → BF happened before MRI recurrence detection;
    + Negative → MRI recurrence detection before BF;
    + NaN  → either `time_to_bf-days` or `rec_mri-time_to_rec-days` is NaN.
* `psa_dt-mri_1-months`: PSA doubling time from baseline to first MRI visit (whether it is positive or not).
* `psa_dt-rec_mri-months`: PSA doubling time from baseline to first positive MRI.

In [ ]:
cols_prev = df.columns
df = features.add_all_features(df)
cols_added = [col for col in list(df.columns) if col not in list(cols_prev)]

In [ ]:
df[[c for c in cols_added if c in df.columns]].describe()

In [ ]:
def print_col_features(df, col):
    df2 = df.copy()
    print(f"{col} Nan: {int(df2[col].isna().sum())}." )
    
for col in cols_added: print_col_features(df,col)

## 8. Excluding subjects with no biopsy data

In [ ]:
df2 = df[df['biopsy-date'].notna()]
df2.shape

## 8. Save cleaned dataset

Saves to `df-clean-02_<timestamp>.csv` and `.xlsx` in the Montreal timezone.

In [ ]:
csv_path, xlsx_path = io.save(df2, stem='df-clean-v1', tz='America/Montreal')
print(f'Final shape: {df2.shape}')
df.head()